<a href="https://colab.research.google.com/github/YASH-HASH-PIXEL/-ANN-Linear_Regression/blob/main/mlops_technology.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import unittest
import sys

# "Application" code to test
def calculate_discount(price: float, discount: float) -> float:
    """Return the final price after applying a discount percentage."""
    if price < 0 or discount < 0:
        raise ValueError("Price and discount must be non-negative")
    return price * (1 - discount / 100)

#  Automated tests
class TestPricing(unittest.TestCase):
    def test_valid_discount(self):
        self.assertAlmostEqual(calculate_discount(100, 20), 80.0)

    def test_zero_discount(self):
        self.assertEqual(calculate_discount(50, 0), 50.0)

    def test_negative_price(self):
        with self.assertRaises(ValueError):
            calculate_discount(-10, 10)

#  Simulated deployment step -
def deploy(version: str = "latest"):
    print(f"Deploying version {version} to production...")
    # In a real pipeline, this would call a cloud API, k8s, etc.
    print("Deployment successful!")

# -Main pipeline runner --
if __name__ == "__main__":
    # 1. Run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestPricing)
    runner = unittest.TextTestRunner(verbosity=1)
    result = runner.run(suite)

    # 2. Gate: if tests fail, stop the pipeline
    if not result.wasSuccessful():
        print("Tests failed. Pipeline aborted.", file=sys.stderr)
        sys.exit(1)

    # 3. All tests passed → proceed with deployment
    deploy(version="1.2.3")

...
----------------------------------------------------------------------
Ran 3 tests in 0.005s

OK


Deploying version 1.2.3 to production...
Deployment successful!


In [ ]:
# app.py – Real‑time CI/CD pipeline using a GitHub webhook
from flask import Flask, request
import subprocess

app = Flask(__name__)

@app.route('/webhook', methods=['POST'])
def pipeline():
    data = request.json
    # Only react to pushes to the main branch
    if data.get('ref') == 'refs/heads/main':
        subprocess.run(['git', 'pull', 'origin', 'main'])
        # Run tests (e.g., pytest)
        if subprocess.run(['python', '-m', 'pytest', 'tests/']).returncode == 0:
            # Build & deploy (e.g., Docker)
            subprocess.run(['docker', 'build', '-t', 'myapp', '.'])
            subprocess.run(['docker', 'run', '-d', '-p', '80:80', 'myapp'])
            return 'Deployed successfully', 200
        else:
            return 'Tests failed', 500
    return 'Ignored', 200

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
